In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import os
from collections import Counter
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay


In [7]:
class PTDataset(Dataset):
    def __init__(self, root_dir, target_size=(500, 500), transform=None):
        """
        Args:
            root_dir (str): Path to the dataset directory (e.g., Training folder).
            target_size (tuple): Desired output size (height, width).
            transform (callable, optional): Optional transformations (on CPU).
        """
        self.root_dir = root_dir
        self.target_size = target_size
        self.transform = transform
        self.classes = ['undetectable', 'low', 'medium', 'high']
        # self.classes = ['0.undetectable', '1.low', '2.medium', '3.high', '4.very high']


        # Collect all file paths and labels
        self.file_list = []
        for label in self.classes:
            class_path = os.path.join(root_dir, label)
            if not os.path.exists(class_path):
                continue  # Skip if folder doesn't exist
            for file in os.listdir(class_path):
                if file.endswith('.pt'):
                    full_path = os.path.join(class_path, file)
                    class_index = self.classes.index(label)
                    self.file_list.append((full_path, class_index))

        # Pre-load everything into memory (CPU)
        self.data_list = []
        for file_path, label in self.file_list:
            # Load from disk to CPU memory
            tensor_data = torch.load(file_path, map_location='cpu')  # [C, T, H, W]

            # Ensure enough frames
            max_frames = tensor_data.shape[1]
            # selected_frame_indices = [ 49, 59, 69, 79, 89, 99, 109, 119, 129, 139, 149, 159, 169, 179]
            selected_frame_indices = [69, 89, 109, 129, 149, 179]
            selected_frame_indices = [i for i in selected_frame_indices if i < max_frames]
            if len(selected_frame_indices) < 6:
                raise ValueError(f"Not enough frames in {file_path}, available: {max_frames}, required: 180")

            # Compute average of the first 20 frames
            avg_first_20 = torch.mean(tensor_data[:, :20, :, :], dim=1, keepdim=True)  # [C, 1, H, W]
            selected_frames = tensor_data[:, selected_frame_indices, :, :]             # [C, 6, H, W]

            # Concatenate to form a 7-frame tensor
            final_tensor = torch.cat((avg_first_20, selected_frames), dim=1)  # [C, 7, H, W]
            final_tensor = final_tensor.squeeze(0) if final_tensor.shape[0] == 1 else final_tensor

            # Resize on CPU
            if final_tensor.dim() == 3:
                # shape [7, H, W]
                final_tensor = final_tensor.unsqueeze(0)  # -> [1, 7, H, W]

            resized_tensor = F.interpolate(
                final_tensor,
                size=self.target_size,
                mode='bilinear',
                align_corners=False
            )

            # Optional transform
            if self.transform:
                resized_tensor = self.transform(resized_tensor)

            # Model expects input_channels=7, flatten [C=1, frames=7, H, W] -> [7, H, W]
            if resized_tensor.shape[0] == 1:
                resized_tensor = resized_tensor.squeeze(0)  # shape [7, H, W]

            # Store (tensor, label)
            self.data_list.append((resized_tensor, label))

    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, idx):
        return self.data_list[idx]

In [ ]:
def get_resnet_model(num_classes=4, input_channels=7, dropout_rate=0.243493213909431):
    """
    Build ResNet18 with a custom first conv layer
    that expects `input_channels` and adds a Dropout layer.

    model_depth = 18 (ResNet18)
    """
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    
    # Replace first conv to match your input_channels
    model.conv1 = nn.Conv2d(input_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)
    
    # Replace FC layer to include Dropout before classification
    model.fc = nn.Sequential(
        nn.Dropout(p=dropout_rate),  # Dropout before final classification
        nn.Linear(model.fc.in_features, num_classes)
    )
    return model


def evaluate_model(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct, total = 0, 0

    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            # Use AMP if on GPU
            with torch.amp.autocast(device_type='cuda', enabled=(device.type == 'cuda')):
                outputs = model(inputs)
                loss = criterion(outputs, labels)

            running_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_loss = running_loss / total
    avg_acc = 100.0 * correct / total
    return avg_loss, avg_acc


def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, device, num_epochs=25):
    """
    Basic training routine using CrossEntropyLoss
    for single-label, multi-class classification.
    """
    
    scaler = torch.amp.GradScaler(enabled=(device.type == 'cuda'))

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct, total = 0, 0

        for inputs, labels in train_loader:
            inputs = inputs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad()

            
            with torch.amp.autocast(device_type='cuda', enabled=(device.type == 'cuda')):
                outputs = model(inputs)
                loss = criterion(outputs, labels)

            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        epoch_loss = running_loss / total
        epoch_acc = 100.0 * correct / total

        val_loss, val_acc = evaluate_model(model, val_loader, criterion, device)
        scheduler.step()

        print(f"Epoch [{epoch+1}/{num_epochs}] "
              f"Train Loss: {epoch_loss:.4f}, Train Acc: {epoch_acc:.2f}%, "
              f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")

    print("Training complete.")

def plot_confusion_matrix(model, loader, device, class_names):
    """
    Generates and displays a confusion matrix for the model on the given loader.
    """
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    conf_matrix = confusion_matrix(all_labels, all_preds)
    disp = ConfusionMatrixDisplay(confusion_matrix=conf_matrix, display_labels=class_names)
    disp.plot(cmap=plt.cm.Blues)
    plt.title("Confusion Matrix")
    plt.show()


train_dataset_path = 'H:/Datasets/int_split/Training/'
val_dataset_path   = 'H:/Datasets/int_split/Validation/'
test_dataset_path  = 'H:/Datasets/int_split/Testing/'

# train_dataset_path = 'H:/Datasets/Log_split/Training/'
# val_dataset_path   = 'H:/Datasets/Log_split/Validation/'
# test_dataset_path  = 'H:/Datasets/Log_split/Testing/'

def main():
    #Check device for CUDA or CPU
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    
    train_dataset = PTDataset(root_dir=train_dataset_path, target_size=(500, 500))
    val_dataset   = PTDataset(root_dir=val_dataset_path,   target_size=(500, 500))
    test_dataset  = PTDataset(root_dir=test_dataset_path,  target_size=(500, 500))

   
    train_labels = [label for _, label in train_dataset.data_list]
    print("Labels in dataset:", set(train_labels))

    class_counts = Counter(train_labels)
    weights = [1.0 / class_counts[label] for label in train_labels]
    
    train_sampler = WeightedRandomSampler(
        weights=weights,
        num_samples=len(weights),
        replacement=True
    )

    
    use_pin_memory = (device.type == 'cuda')
    batch_size = 32 
    
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        sampler=train_sampler,
        shuffle=False,
        num_workers=0,
        pin_memory=use_pin_memory
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=use_pin_memory
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=use_pin_memory
    )

    
    best_val_acc = -float('inf')
    best_model_state = None

    # Train/Evaluate 5 times
    for run_idx in range(5):
        print(f"\n=== Training Run {run_idx+1} of 5 ===")

        
        model = get_resnet_model(num_classes=4, input_channels=7)  
        model.to(device)

        criterion = nn.CrossEntropyLoss()

        learning_rate = 0.0038825206157311557
        weight_decay  = 0.0001931053552153856
        gamma_rate    = 0.9388047294838997

        
        optimizer = optim.SGD(
            model.parameters(),
            lr=learning_rate,
            weight_decay=weight_decay,
            momentum=0.9
        )

        scheduler = optim.lr_scheduler.ExponentialLR(optimizer=optimizer, gamma=gamma_rate)

        
        num_epochs = 25
        train_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            criterion=criterion,
            optimizer=optimizer,
            scheduler=scheduler,
            device=device,
            num_epochs=num_epochs
        )

        # Evaluate on validation set
        val_loss, val_acc = evaluate_model(model, val_loader, criterion, device)
        print(f"Run {run_idx+1} validation accuracy: {val_acc:.2f}%")

        # Keep track of best model so far
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = model.state_dict()
            print(f"New best model found with val_acc={val_acc:.2f}% (Run {run_idx+1}).")

    # After all 5 runs, save only the best model
    if best_model_state is not None:
        torch.save(best_model_state, 'Clinical_model.pth')
        print(f"\nBest model saved with val_acc={best_val_acc:.2f}%")

        best_model = get_resnet_model(num_classes=4, input_channels=7)
        best_model.load_state_dict(best_model_state)
        best_model.to(device)

        test_loss, test_acc = evaluate_model(best_model, test_loader, criterion, device)
        print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_acc:.2f}% for the best model")

if __name__ == "__main__":
    main()


Using device: cuda
Labels in dataset: {0, 1, 2, 3}

=== Training Run 1 of 5 ===
Epoch [1/18] Train Loss: 1.3560, Train Acc: 41.84%, Val Loss: 1.2972, Val Acc: 59.18%
Epoch [2/18] Train Loss: 0.8087, Train Acc: 70.21%, Val Loss: 0.7574, Val Acc: 73.47%
Epoch [3/18] Train Loss: 0.5634, Train Acc: 75.89%, Val Loss: 0.6552, Val Acc: 83.67%
Epoch [4/18] Train Loss: 0.2702, Train Acc: 89.36%, Val Loss: 0.5988, Val Acc: 85.71%
Epoch [5/18] Train Loss: 0.3153, Train Acc: 87.94%, Val Loss: 0.7486, Val Acc: 79.59%
Epoch [6/18] Train Loss: nan, Train Acc: 60.28%, Val Loss: nan, Val Acc: 16.33%
Epoch [7/18] Train Loss: 0.1849, Train Acc: 94.33%, Val Loss: nan, Val Acc: 16.33%
Epoch [8/18] Train Loss: 0.2362, Train Acc: 90.78%, Val Loss: nan, Val Acc: 16.33%
Epoch [9/18] Train Loss: 0.0870, Train Acc: 96.45%, Val Loss: nan, Val Acc: 16.33%
Epoch [10/18] Train Loss: nan, Train Acc: 78.01%, Val Loss: nan, Val Acc: 16.33%
Epoch [11/18] Train Loss: 0.1354, Train Acc: 95.74%, Val Loss: nan, Val Acc: 16.

In [14]:
import torch
import torchvision.models as models
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
plt.rcParams['font.family'] = 'Arial'


def load_best_model(model_path, num_classes=4, input_channels=7, dropout_rate=0.243493213909431):
    """
    Load the best trained model from saved weights.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    model.conv1 = torch.nn.Conv2d(input_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)
    model.fc = torch.nn.Sequential(
        torch.nn.Dropout(p=dropout_rate),
        torch.nn.Linear(model.fc.in_features, num_classes)
    )

    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()  

    return model, device

def plot_confusion_matrix(model, loader, device, class_names):
    """
    Generates and displays a confusion matrix for the model on the given loader.
    """
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    # Compute confusion matrix
    conf_matrix = confusion_matrix(all_labels, all_preds)
    disp = ConfusionMatrixDisplay(confusion_matrix=conf_matrix, display_labels=class_names)
    
    fig, ax = plt.subplots(figsize=(8, 6))
    disp.plot(cmap=plt.cm.Blues, ax=ax, colorbar=False)  # no frequency bar
    ax.set_xlabel("Predicted HIV viral load", fontsize=12)
    ax.set_ylabel("True HIV viral load", fontsize=12)

    # Set tick label font to Arial explicitly
    ax.set_xticklabels(class_names, fontname='Arial')
    ax.set_yticklabels(class_names, fontname='Arial')

    plt.show()

model_path = r"D:\ANSA\resnet_model_18_10_true_15.pth"
model, device = load_best_model(model_path)

test_dataset_path = r'D:/Datasets/int_split/Testing/'
test_dataset = PTDataset(root_dir=test_dataset_path, target_size=(500, 500))

test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=32, shuffle=False, num_workers=0, pin_memory=(device.type == 'cuda')
)

class_names = ['Und.', 'Low', 'Med', 'High']  

plot_confusion_matrix(model, test_loader, device, class_names)



RuntimeError: Error(s) in loading state_dict for ResNet:
	Missing key(s) in state_dict: "fc.1.weight", "fc.1.bias". 
	Unexpected key(s) in state_dict: "fc.weight", "fc.bias". 
	size mismatch for conv1.weight: copying a param with shape torch.Size([64, 15, 7, 7]) from checkpoint, the shape in current model is torch.Size([64, 7, 7, 7]).

# Optuna

In [15]:
import os
import json
import torch
import torch.nn as nn
import torch.optim as optim
import optuna
from torch.utils.data import DataLoader
import torchvision.models as models
import numpy as np

# Load datasets
# train_dataset_path = 'H:/Datasets/int_split/Training/'
# val_dataset_path = 'H:/Datasets/int_split/Validation/'
# test_dataset_path = 'H:/Datasets/int_split/Testing/'

train_dataset_path = 'H:/Datasets/Log_split/Training/'
val_dataset_path   = 'H:/Datasets/Log_split/Validation/'
test_dataset_path  = 'H:/Datasets/Log_split/Testing/'

train_dataset = PTDataset(root_dir=train_dataset_path, target_size=(500, 500))
val_dataset = PTDataset(root_dir=val_dataset_path, target_size=(500, 500))
test_dataset = PTDataset(root_dir=test_dataset_path, target_size=(500, 500))

def get_resnet_model(model_depth=34, dropout=0.5, num_classes=4, input_channels=7):
    """Returns a ResNet model with specified depth and dropout."""
    if model_depth == 18:
        model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    elif model_depth == 34:
        model = models.resnet34(weights=models.ResNet34_Weights.DEFAULT)
    elif model_depth == 50:
        model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    else:
        raise ValueError("Invalid ResNet depth. Choose from 18, 34, or 50.")
    
    model.conv1 = nn.Conv2d(input_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)  # Modify for 7 channels
    model.fc = nn.Sequential(
        nn.Dropout(dropout),
        nn.Linear(model.fc.in_features, num_classes)
    )
    return model

def evaluate_model(model, loader, criterion, device):
    """Evaluates the model on a dataset."""
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    avg_loss = running_loss / total
    avg_acc = 100.0 * correct / total
    return avg_loss, avg_acc

def objective(trial):
    """Optuna optimization function with pruning."""
    params = {
        'model_depth': trial.suggest_categorical('model_depth', [18, 34]),
        'learning_rate': trial.suggest_loguniform('learning_rate', 1e-5, 1e-2),
        'weight_decay': trial.suggest_loguniform('weight_decay', 1e-6, 1e-2),
        'dropout': trial.suggest_uniform('dropout', 0, 0.9),
        'gamma': trial.suggest_categorical('gamma', [True, False]),
        'gamma_rate': trial.suggest_uniform('gamma_rate', 0.9, 0.99),
        'optimizer': trial.suggest_categorical('optimizer', ['Adam', 'SGD', 'ASGD', 'LBFGS'])
    }
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = get_resnet_model(params['model_depth'], params['dropout']).to(device)
    criterion = nn.CrossEntropyLoss()
    
    if params['optimizer'] == 'Adam':
        optimizer = optim.Adam(model.parameters(), lr=params['learning_rate'], weight_decay=params['weight_decay'])
    elif params['optimizer'] == 'SGD':
        optimizer = optim.SGD(model.parameters(), lr=params['learning_rate'], weight_decay=params['weight_decay'], momentum=0.9)
    elif params['optimizer'] == 'ASGD':
        optimizer = optim.ASGD(model.parameters(), lr=params['learning_rate'], weight_decay=params['weight_decay'])
    elif params['optimizer'] == 'LBFGS':
        optimizer = optim.LBFGS(model.parameters(), lr=params['learning_rate'])  # Removed weight_decay
    else:
        raise ValueError("Unsupported optimizer type")
    
    scheduler = optim.lr_scheduler.ExponentialLR(optimizer, gamma=params['gamma_rate']) if params['gamma'] else None
    
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
    
    # Training loop with pruning
    for epoch in range(25):
        model.train()
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            def closure():
                optimizer.zero_grad()
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                loss.backward()
                return loss
            
            if params['optimizer'] == 'LBFGS':
                optimizer.step(closure)
            else:
                optimizer.zero_grad()
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
        
        if scheduler:
            scheduler.step()

        # Evaluate on validation set
        val_loss, val_acc = evaluate_model(model, val_loader, criterion, device)
        
        # Report intermediate result for pruning
        trial.report(val_acc, epoch)

        # Stop early if performance is worse than previous trials at this step
        if trial.should_prune():
            raise optuna.TrialPruned()

    return -val_acc  # Minimizing negative accuracy to maximize positive accuracy

# Run Optuna study with Median Pruner
study = optuna.create_study(direction="minimize", pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3))
study.optimize(objective, n_trials=200)

# Display results
best_trial = study.best_trial
print("\nBest trial:")
print(f"  Value: {best_trial.value}")  # Converting back to positive accuracy
print("  Params: ")
for key, value in best_trial.params.items():
    print(f"    {key}: {value}")


ModuleNotFoundError: No module named 'optuna'